# Chapter 16 &mdash; The Verifier View, and its Equivalence to the Decider View

**Concept 4 of the Chapter 16 decomposition:** *The Verifier View, and its Equivalence to the Decider View*

A certificate encodes the nondeterministic choices, so guessing and checking are interchangeable.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-Verifier-View/Concept-Verifier-View.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The second definition of NP, and the one you use in practice:

> $L \in NP$ iff there is a **polynomial-time verifier** $V$ and a polynomial $p$ such
> that $x \in L \iff \exists c,\ |c| \le p(|x|),\ V(x,c)$ accepts.

The two definitions are **equivalent**, and the translation is mechanical:

* **NDTM $\to$ verifier.** The certificate is the **sequence of choices** the NDTM
  makes. The verifier replays them deterministically. The path is polynomial, so the
  certificate is too.
* **Verifier $\to$ NDTM.** Nondeterministically **guess** the certificate symbol by
  symbol, then run $V$.

Guessing and checking are the same thing seen from two sides, and the verifier side is
usually the easier one to write down.

## 2. Definitions

### A verifier and the NDTM it corresponds to

In [ ]:
# --- a tiny CNF toolkit -------------------------------------------------
# A literal is an int: 3 means x3, -3 means NOT x3.
# A clause is a tuple of literals; a formula is a list of clauses.
from itertools import product

def nvars(F):
    return max((abs(l) for c in F for l in c), default=0)

def evaluate(F, assign):
    # assign: dict var -> bool
    return all(any(assign[abs(l)] == (l > 0) for l in c) for c in F)

def brute_sat(F):
    n = nvars(F)
    for bits in product([False, True], repeat=n):
        a = {i + 1: bits[i] for i in range(n)}
        if evaluate(F, a): return a
    return None

def show_cnf(F):
    def lit(l): return ("x%d" % l) if l > 0 else ("~x%d" % -l)
    return " AND ".join("(" + " OR ".join(lit(l) for l in c) + ")" for c in F)


def verify_sat(F, cert):
    # cert is a bit string, one bit per variable -- the CERTIFICATE
    n = nvars(F)
    if len(cert) != n: return False
    a = {i + 1: (cert[i] == '1') for i in range(n)}
    return evaluate(F, a)

def ndtm_choices(F):
    # the NDTM guesses one bit per variable: |choices| = n, tree = 2^n
    return nvars(F)

### Translating between the two views

In [ ]:
def nondeterministic_search(F):
    # explore the whole tree, recording the winning choice sequence
    n = nvars(F)
    for bits in product('01', repeat=n):
        c = ''.join(bits)
        if verify_sat(F, c): return c
    return None

## 3. Tests

A formula, a certificate, and the verifier.

In [ ]:
F = [(1, 2), (-1, 3), (-2, -3)]
print("formula :", show_cnf(F))
cert = nondeterministic_search(F)
print("certificate found :", cert)
print("verifier accepts it? ", verify_sat(F, cert))
assert cert and verify_sat(F, cert)

The certificate is **short** &mdash; one bit per variable.

In [ ]:
print("|x| (clauses)     :", len(F))
print("|c| (certificate) :", len(cert), " = number of variables")
assert len(cert) <= 3 * nvars(F)
print("\npolynomial in the input size, as the definition requires")

**Wrong certificates are rejected**, which is the other half of the definition.

In [ ]:
for bad in ['000', '011', '110']:
    print("  cert %-5s accepted? %s" % (bad, verify_sat(F, bad)))
allcerts = [''.join(b) for b in product('01', repeat=3)]
good = [c_ for c_ in allcerts if verify_sat(F, c_)]
print("\naccepting certificates :", good)
assert good and len(good) < len(allcerts)

An **unsatisfiable** formula has no certificate at all.

In [ ]:
U = [(1,), (-1,)]
print("formula :", show_cnf(U))
print("any certificate? ", nondeterministic_search(U))
assert nondeterministic_search(U) is None
print("\nSo U is not in SAT -- and note the verifier never has to say WHY.")

**The certificate IS the choice sequence.** That is the equivalence.

In [ ]:
print("NDTM view : guess bit for x1, guess bit for x2, ..., then evaluate")
print("verifier  : receive 'x1x2x3' as the certificate, then evaluate")
print()
print("choices made :", ndtm_choices(F), " certificate length :", len(cert))
assert ndtm_choices(F) == len(cert)
print()
print("Same bits, same order.  One machine guesses them; the other is handed them.")

Why the verifier view is the practical one.

In [ ]:
print("To show a problem is in NP, you do NOT design an NDTM.")
print("You answer two questions:")
print("   1. what would a solution look like?   (the certificate)")
print("   2. how do I check a claimed one?      (the verifier)")
print()
print("If both answers are polynomial, the problem is in NP.  Done.")

## 4. Exercises


1. Give certificate and verifier for "this graph is 3-colourable".
2. Why must the certificate be **polynomially bounded** and not merely finite?
3. What is the certificate for a problem in $P$?

In [ ]:
# Your work for the exercises above.